In [ ]:
import numpy              as np
import matplotlib.pyplot  as plt
import seaborn            as sns
import os

from pymatgen.io.vasp.outputs import Vasprun
from pymatgen.io.vasp.outputs import Locpot

from libraries import alignment, vasp_files

sns.set_theme()

First we compute the average electrostatic potential of the bulk and, for each slab, in a bulk-like region and in the vacuum region of the
planar-averaged potential; then we compute the band alignment of each slab. The position of the valence band maximum with respect to the vacuum level is

\begin{equation}
    VBM_{vac} = VBM_{DFT} + \left(\bar{V}_{bulk\text{-}like} - \bar{V}_{bulk}\right) - \bar{V}_{vac}, \qquad CBM_{vac} = VBM_{vac} + E_{gap}
\end{equation}

where $VBM_{DFT}$ and $E_{gap}$ come from the bulk band structure, $\bar{V}_{bulk}$ is the average potential of the separate bulk
calculation, and $\bar{V}_{bulk\text{-}like}$, $\bar{V}_{vac}$ are the mean planar-averaged potentials in the bulk-like and vacuum regions of the slab.
The ionization potential and electron affinity are $IP = -VBM_{vac}$ and $EA = -CBM_{vac}$.

The formula is implemented in `alignment.get_band_alignment` (`libraries/alignment.py`), which also locates the bulk-like and
vacuum regions (`region_mode`) and returns the ionization potential and electron affinity.

This step needs DFT data (`LOCPOT` and `vasprun.xml`) for the bulk and for every slab, in the `alignment/` folder of each
(`<folder>/alignment/LOCPOT`); its results are written there too.

In [ ]:
# Define name of reference folder which stores the energies of each slab
slab_folder = '/Users/cibran/Desktop/revision/BiSI-NUPDOWN/alignments/NUPDOWN=2'

# Ratio to discard from the edges for averaging
discard_ratio = 0.25

# How the bulk-like and vacuum regions of the slab are located along the surface normal (c axis):
#   'midpoint': legacy; the cell is cut in two halves, the slab is ASSUMED to sit in the first half (swaps both regions otherwise)
#   'auto'    : located from the atomic positions (slab = atoms, vacuum = largest gap); valid wherever the slab sits in the cell
region_mode = 'auto'

# Bulk

In [ ]:
bulk_folder = f'{slab_folder}/bulk'
bulk_align  = vasp_files.alignment_dir(bulk_folder)

locpot = Locpot.from_file(f'{bulk_align}/LOCPOT')
linear_mac_pot = locpot.get_average_along_axis(2)

In [ ]:
# Load vasprun
vasprun = Vasprun(f'{bulk_align}/vasprun.xml')

# Get band structure
band_structure = vasprun.get_band_structure()

# Get valence band maximum and band gap
valence_band_DFT = band_structure.get_vbm()["energy"]
band_gap         = band_structure.get_band_gap()["energy"]

In [ ]:
mac_avg_pot_bulk = np.mean(linear_mac_pot)

np.savetxt(f'{bulk_align}/macroscopic_average_potential', [mac_avg_pot_bulk])

print(f'Mean value: {mac_avg_pot_bulk}')

In [ ]:
x_label = np.arange(len(linear_mac_pot)) * vasprun.final_structure.lattice.c / len(linear_mac_pot)
plt.plot(x_label, linear_mac_pot, label='Linear average potential')
plt.plot([x_label[0], x_label[-1]], [mac_avg_pot_bulk, mac_avg_pot_bulk], ':r', label='Mean value')

plt.savefig(f'{bulk_align}/macroscopic_average_potential.pdf', bbox_inches='tight')
plt.show()

# Slab

In [ ]:
# Iterate over slabs
for miller_index_str in os.listdir(slab_folder):
    # Define current folder
    miller_folder = f'{slab_folder}/{miller_index_str}'

    # Skip in case it is not a folder or it is the bulk folder
    if (not os.path.isdir(miller_folder)) or (miller_index_str == 'bulk'):
        continue

    print(miller_folder)

    miller_align = vasp_files.alignment_dir(miller_folder)
    if not os.path.isdir(miller_align):
        print('\tno alignment/ folder: skipped')
        continue

    # Load planar average potential along the surface normal
    locpot = Locpot.from_file(f'{miller_align}/LOCPOT')
    linear_mac_pot = locpot.get_average_along_axis(2)

    vasprun = Vasprun(f'{miller_align}/vasprun.xml')

    # Band edges with respect to the vacuum level (the bulk-like and vacuum regions of the slab are located inside)
    alignments = alignment.get_band_alignment(linear_mac_pot, vasprun.final_structure,
                                              vbm_dft=valence_band_DFT, band_gap=band_gap,
                                              bulk_average_potential=mac_avg_pot_bulk,
                                              discard_ratio=discard_ratio, mode=region_mode)
    (b0, b1), (v0, v1) = alignments.bulk_region, alignments.vacuum_region

    x_label = np.arange(len(alignments.potential)) * vasprun.final_structure.lattice.c / len(alignments.potential)
    plt.plot(x_label, alignments.potential, label='Slab')

    # Bulk-like region
    plt.plot([x_label[b0], x_label[b1 - 1]], [alignments.bulk_like_potential] * 2, ':r', label='Bulk')

    # Vacuum region
    plt.plot([x_label[v0], x_label[v1 - 1]], [alignments.vacuum_potential] * 2, ':r', label='Vacuum')

    # Save the data
    np.savetxt(f'{miller_align}/macroscopic_average_potential', [alignments.bulk_like_potential, alignments.vacuum_potential])
    np.savetxt(f'{miller_align}/valence_band',    [alignments.valence_band])
    np.savetxt(f'{miller_align}/conduction_band', [alignments.conduction_band])

    print(f'\tValence band: {alignments.valence_band}')
    print(f'\tConduction band: {alignments.conduction_band}')
    print(f'\tIonization potential: {alignments.ionization_potential}')
    print(f'\tElectron affinity: {alignments.electron_affinity}')

    plt.savefig(f'{miller_align}/macroscopic_average_potential.pdf', bbox_inches='tight')
    plt.show()